# Final POI recommendation baseline
Run with the project installed, using `uv sync`. Binary labels stay unchanged; individual scores average the active preferences. GFAR aggregates the candidate union.


In [ ]:
from ourney_travelsync import FEATURE_NAMES, load_places, AverageWithoutMisery
from ourney_travelsync.recommenders import calculate_preference_score, prepare_group, recommend_group

user = dict(zip(FEATURE_NAMES, [4, 5, 3, 2, 2, 3, 3, 3, 4, 1]))
example_poi = dict.fromkeys(FEATURE_NAMES, 0) | {"food": 1, "cafe": 1}
print(calculate_preference_score(user, example_poi))  # 4.5


In [ ]:
places = load_places()
users = {
    "alice": user,
    "bob": dict(zip(FEATURE_NAMES, [3, 2, 5, 4, 3, 2, 4, 2, 5, 3])),
}
group = prepare_group(places, users, individual_top_n=100)
print("POIs:", len(places.place_ids), "Candidates:", len(group.poi_ids))
print("Shapes:", group.user_features.shape, group.poi_features.shape, group.individual_scores.shape)
metadata = {row["place_id"]: row for row in places.records}
for result in recommend_group(places, users, top_k=10):
    print(result.poi_id, metadata[result.poi_id]["place_name"], result.score)


In [ ]:
awm_results = recommend_group(places, users, recommender=AverageWithoutMisery(2.5))
for result in awm_results:
    print(result.poi_id, metadata[result.poi_id]["place_name"], result.score)


In [ ]:
gfar_results = recommend_group(places, users)
for result in gfar_results:
    print(result.poi_id, metadata[result.poi_id]["place_name"], result.score)


## AWM vs GFAR: shared satisfaction evaluation

Both algorithms use the same users, candidate pool (union of each member's top 100), and requested top-K. Evaluation uses the original 1-5 individual preference scores, not AWM's mean output or GFAR's marginal gain.

- **Mean satisfaction:** average of each member's mean score across the returned POIs.
- **Minimum member satisfaction:** lowest of those member averages, not the lowest single POI score.
- **Returned POIs:** reported because AWM may return fewer than K; empty lists have undefined satisfaction (N/A).

These metrics evaluate agreement with the current category-based scoring assumptions. Actual recommendation quality needs user feedback. A higher minimum member satisfaction means the least-satisfied member has a higher average predicted score; it does not guarantee every place is acceptable.


In [ ]:
import numpy as np
from ourney_travelsync import GFAR

comparison_top_k = 10
comparison_candidate_top_n = 100
comparison_group = prepare_group(
    places, users, individual_top_n=comparison_candidate_top_n,
)
comparison_group.validate(require_scores=True)

# Generate both lists from exactly the same scored candidate pool.
comparison_results = {
    "AWM": AverageWithoutMisery(2.5).recommend(comparison_group, top_k=comparison_top_k),
    "GFAR": GFAR().recommend(comparison_group, top_k=comparison_top_k),
}


from ourney_travelsync.evaluation import evaluate_satisfaction

comparison_metrics = {
    name: evaluate_satisfaction(comparison_group, results)
    for name, results in comparison_results.items()
}


def format_satisfaction(value):
    return "N/A" if value is None else f"{value:.3f}"


print(f"Shared candidates: {len(comparison_group.poi_ids)} | Requested top-K: {comparison_top_k}")
print("Satisfaction values use the same 1-5 preference scale. Higher is better.")
print(f"{'Algorithm':<12} {'Returned POIs':>14} {'Mean satisfaction':>20} {'Minimum member satisfaction':>30}")
for name, metrics in comparison_metrics.items():
    print(f"{name:<12} {metrics['returned_pois']:>14} "
          f"{format_satisfaction(metrics['mean_satisfaction']):>20} "
          f"{format_satisfaction(metrics['minimum_member_satisfaction']):>30}")

print("\nEach member's average score across the returned POIs:")
print(f"{'Member':<20} {'AWM':>10} {'GFAR':>10}")
for member in comparison_group.user_ids:
    awm_value = comparison_metrics['AWM']['member_satisfaction'][member]
    gfar_value = comparison_metrics['GFAR']['member_satisfaction'][member]
    print(f"{member:<20} {format_satisfaction(awm_value):>10} {format_satisfaction(gfar_value):>10}")
